# 01 — Data Exploration

**OandaFX** — Explore raw OANDA candle data, check data quality, analyze spreads & volume.

Run this notebook after downloading raw data (via `scripts/download_history.py` or directly from OANDA).
Data is loaded from Google Drive for Colab persistence.

In [ ]:
# ─── Mount Google Drive & Setup ─────────────────────────────────
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive/oandafx'
    # Clone repo if needed
    if not os.path.exists('/content/oandafx'):
        !git clone https://github.com/your-org/oandafx.git /content/oandafx
    sys.path.insert(0, '/content/oandafx')
else:
    DRIVE_BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))
    sys.path.insert(0, DRIVE_BASE)

DATA_DIR = os.path.join(DRIVE_BASE, 'data', 'raw')
print(f'Data directory: {DATA_DIR}')
print(f'Exists: {os.path.exists(DATA_DIR)}')

In [ ]:
# ─── Imports ─────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data.storage import load_candles, detect_gaps, detect_anomalies

plt.style.use('dark_background')
sns.set_palette('Set2')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.5f}'.format)

In [ ]:
# ─── Discover Available Data ─────────────────────────────────────
data_path = Path(DATA_DIR)
pairs = sorted([d.name for d in data_path.iterdir() if d.is_dir()]) if data_path.exists() else []
print(f'Pairs found: {pairs}\n')

# Inventory
inventory = []
for pair in pairs:
    pair_dir = data_path / pair
    for f in sorted(pair_dir.glob('*.parquet')):
        df = load_candles(f)
        inventory.append({
            'Pair': pair,
            'Timeframe': f.stem,
            'Bars': len(df),
            'Start': df['time'].min() if not df.empty else None,
            'End': df['time'].max() if not df.empty else None,
        })

inv_df = pd.DataFrame(inventory)
display(inv_df)

## Price Visualization

In [ ]:
# ─── Plot price series for each pair (Daily) ────────────────────
fig, axes = plt.subplots(len(pairs), 1, figsize=(16, 4 * len(pairs)), sharex=False)
if len(pairs) == 1:
    axes = [axes]

for ax, pair in zip(axes, pairs):
    path = data_path / pair / 'D.parquet'
    if not path.exists():
        continue
    df = load_candles(path)
    ax.plot(df['time'], df['close'], linewidth=0.7)
    ax.set_title(f'{pair} — Daily Close', fontsize=13)
    ax.set_ylabel('Price')
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## Data Quality Checks

In [ ]:
# ─── Gap Detection ───────────────────────────────────────────────
GRANULARITY_SECONDS = {'M15': 900, 'H1': 3600, 'H4': 14400, 'D': 86400}

for pair in pairs:
    for tf, secs in GRANULARITY_SECONDS.items():
        path = data_path / pair / f'{tf}.parquet'
        if not path.exists():
            continue
        df = load_candles(path)
        gaps = detect_gaps(df, secs)
        if not gaps.empty:
            print(f'\n⚠️  {pair}/{tf}: {len(gaps)} gaps detected')
            display(gaps.head(5))
        else:
            print(f'✅ {pair}/{tf}: No gaps')

In [ ]:
# ─── Anomaly Detection (bars with range > 5× ATR) ────────────────
for pair in pairs[:3]:  # Check first 3 pairs
    path = data_path / pair / 'M15.parquet'
    if not path.exists():
        continue
    df = load_candles(path)
    anomalies = detect_anomalies(df, atr_multiplier=5.0)
    if not anomalies.empty:
        print(f'\n⚠️  {pair}/M15: {len(anomalies)} anomalous bars')
        display(anomalies[['time', 'open', 'high', 'low', 'close', 'bar_range', 'atr_20']].head(5))
    else:
        print(f'✅ {pair}/M15: No anomalies')

## Spread Analysis

In [ ]:
# ─── Spread Distribution ─────────────────────────────────────────
pair = pairs[0] if pairs else 'EUR_USD'
df = load_candles(data_path / pair / 'M15.parquet')

if 'bid_close' in df.columns and 'ask_close' in df.columns:
    df['spread'] = df['ask_close'] - df['bid_close']
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.hist(df['spread'].dropna(), bins=100, alpha=0.7, color='cyan')
    ax1.set_title(f'{pair} — Spread Distribution (M15)')
    ax1.set_xlabel('Spread (price units)')
    ax1.axvline(df['spread'].median(), color='red', linestyle='--', label=f'Median: {df["spread"].median():.6f}')
    ax1.legend()
    
    # Spread by hour of day
    df['hour'] = pd.to_datetime(df['time']).dt.hour
    hourly_spread = df.groupby('hour')['spread'].mean()
    ax2.bar(hourly_spread.index, hourly_spread.values, color='cyan', alpha=0.7)
    ax2.set_title(f'{pair} — Average Spread by Hour (UTC)')
    ax2.set_xlabel('Hour (UTC)')
    
    plt.tight_layout()
    plt.show()
else:
    print('No bid/ask data available — download with price=BA')

## Volume & Pair Correlations

In [ ]:
# ─── Pair Return Correlations ────────────────────────────────────
returns = {}
for pair in pairs:
    path = data_path / pair / 'D.parquet'
    if not path.exists():
        continue
    df = load_candles(path)
    if not df.empty:
        returns[pair] = np.log(df['close'] / df['close'].shift(1)).values[1:]

if returns:
    min_len = min(len(v) for v in returns.values())
    ret_df = pd.DataFrame({k: v[:min_len] for k, v in returns.items()})
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(ret_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
    ax.set_title('Daily Return Correlations')
    plt.tight_layout()
    plt.show()

In [ ]:
# ─── Save Summary to Drive ───────────────────────────────────────
summary_path = os.path.join(DRIVE_BASE, 'data', 'data_exploration_summary.csv')
os.makedirs(os.path.dirname(summary_path), exist_ok=True)
inv_df.to_csv(summary_path, index=False)
print(f'Summary saved to {summary_path}')